# shallow_vs_deep_copy
**Prerequisites:** memory_model, references, mutability

**Outcome:** After this notebook you will know exactly what shallow copy and deep copy produce at the memory level, when each one is correct, and how to choose between them without guessing.

## The Problem

In [ ]:
import copy

original = {
    "pipeline": "etl",
    "stages": ["extract", "transform", "load"]
}

cloned = copy.copy(original)
cloned["pipeline"] = "streaming"     # change the string
cloned["stages"].append("validate")  # change the nested list

print(original)
# pipeline is unchanged — good
# but stages has 'validate' in it — why did the original change?

## The Concept

A shallow copy creates a new container object but fills it with references to the same inner objects as the original. The outer shell is independent. Everything inside is shared. A deep copy creates a new container and recursively creates new copies of every object inside it, all the way down. Nothing is shared. The cost of a deep copy is proportional to the size and depth of the object graph. The cost of the wrong choice is a silent mutation bug that only appears under specific call sequences.

## Minimal Example

In [ ]:
import copy

original = [[1, 2], [3, 4]]

shallow = copy.copy(original)
deep    = copy.deepcopy(original)

original[0].append(99)

print(original)  # [[1, 2, 99], [3, 4]]
print(shallow)   # [[1, 2, 99], [3, 4]] — inner list shared
print(deep)      # [[1, 2],     [3, 4]] — fully independent

## How Python Does It

Shallow copy allocates a new container and copies each element reference from the original into the new container — one pass, O(n) where n is the number of top-level elements. Deep copy walks the entire object graph using a memo dictionary to track already-copied objects, which prevents infinite loops on circular references. Every object encountered that has not yet been copied gets a new copy created and registered in the memo dict before recursing into its children.

In [ ]:
import copy

original = [["a", "b"], ["c", "d"]]
shallow  = copy.copy(original)
deep     = copy.deepcopy(original)

# outer containers are different objects
print(original is shallow)         # False
print(original is deep)            # False

# inner lists: shallow shares them, deep does not
print(original[0] is shallow[0])   # True  — same inner object
print(original[0] is deep[0])      # False — independent copy

## Building Up

In [ ]:
# three ways to shallow copy a list
jobs = ["job_1", "job_2", "job_3"]

via_slice      = jobs[:]
via_copy       = jobs.copy()
via_list       = list(jobs)
via_copy_module = __import__("copy").copy(jobs)

print(via_slice is jobs)       # False — all are new lists
print(via_copy is jobs)        # False
print(via_list is jobs)        # False

# for flat lists of immutables all four are equivalent in effect
jobs.append("job_4")
print(via_slice)   # unchanged — ["job_1", "job_2", "job_3"]

In [ ]:
# shallow copy of a dict
config = {"host": "localhost", "tags": ["prod", "us-east"]}

via_copy   = config.copy()
via_dict   = dict(config)
via_unpack = {**config}

# all produce independent outer dicts
via_copy["host"] = "remotehost"
print(config["host"])   # localhost — outer key unaffected

# but inner list is still shared
via_copy["tags"].append("eu-west")
print(config["tags"])   # ["prod", "us-east", "eu-west"] — shared

In [ ]:
import copy

# deep copy — nothing shared at any depth
config = {"host": "localhost", "tags": ["prod", "us-east"]}
deep = copy.deepcopy(config)

deep["tags"].append("eu-west")
print(config["tags"])   # ["prod", "us-east"] — completely isolated

In [ ]:
import copy

# deep copy handles circular references correctly
a = {"name": "producer"}
b = {"name": "consumer", "partner": a}
a["partner"] = b   # circular reference

deep = copy.deepcopy(a)   # no infinite loop — memo dict prevents it
print(deep["name"])
print(deep["partner"]["name"])
print(deep["partner"]["partner"] is deep)   # True — cycle preserved in copy

In [ ]:
import copy

# custom __copy__ and __deepcopy__ for classes
class Pipeline:
    def __init__(self, name, stages):
        self.name   = name
        self.stages = stages

    def __copy__(self):
        return Pipeline(self.name, self.stages)   # shallow: shares stages list

    def __deepcopy__(self, memo):
        return Pipeline(self.name, copy.deepcopy(self.stages, memo))

p = Pipeline("etl", ["extract", "transform"])

s = copy.copy(p)
d = copy.deepcopy(p)

print(s.stages is p.stages)   # True  — shallow shares the list
print(d.stages is p.stages)   # False — deep has its own list

## Where It Breaks

In [ ]:
import copy

# shallow copy used on a nested structure — inner mutation leaks
cluster = {
    "us-east": {"api": "running", "worker": "running"},
    "eu-west": {"api": "running", "worker": "stopped"}
}

snapshot = copy.copy(cluster)
snapshot["us-east"]["api"] = "crashed"   # mutates shared inner dict

print(cluster["us-east"]["api"])   # "crashed" — original contaminated

In [ ]:
import copy

# deep copy used unnecessarily on a flat structure — wasted allocation
latencies = [90, 120, 150, 80, 200]   # flat list of immutables

# both produce equivalent results — deep copy is overkill here
shallow = latencies.copy()
deep    = copy.deepcopy(latencies)

# for flat lists of immutables shallow is always sufficient
# deep copy wastes time traversing a structure that has nothing to copy deeply

## The Fix

In [ ]:
import copy

# correct: deep copy for nested mutable structures
cluster = {
    "us-east": {"api": "running", "worker": "running"},
    "eu-west": {"api": "running", "worker": "stopped"}
}

snapshot = copy.deepcopy(cluster)
snapshot["us-east"]["api"] = "crashed"

print(cluster["us-east"]["api"])    # "running" — original safe
print(snapshot["us-east"]["api"])   # "crashed" — only snapshot changed

In [ ]:
# decision rule in code
def safe_copy(obj):
    """
    Use shallow copy when the object is flat or contains only immutables.
    Use deep copy when the object contains nested mutable structures.
    """
    import copy
    if any(isinstance(v, (list, dict, set)) for v in (obj.values() if isinstance(obj, dict) else obj)):
        return copy.deepcopy(obj)
    return obj.copy()

flat   = {"host": "localhost", "port": 5432}
nested = {"host": "localhost", "tags": ["prod"]}

print(safe_copy(flat))
print(safe_copy(nested))

## In a Real System

In [ ]:
import copy

# pipeline stage receives a record and must not mutate the original
# the record contains nested lists so shallow copy is not safe

def transform_stage(record):
    working = copy.deepcopy(record)   # isolate completely
    working["status"] = "transformed"
    working["tags"].append("processed")
    working["metrics"]["duration_ms"] = 42
    return working

original_record = {
    "id": "job_1",
    "status": "pending",
    "tags": ["raw"],
    "metrics": {"rows": 1000}
}

result = transform_stage(original_record)

print(original_record)   # unchanged — safe to retry, audit, or route elsewhere
print(result)            # enriched version

## Performance

Shallow copy is O(n) where n is the number of top-level items. Deep copy is O(m) where m is the total number of objects in the entire graph. For large nested structures deep copy can be significantly slower. In hot paths — tight loops, high-throughput pipelines — prefer designing your data so shallow copy is sufficient, or use immutable structures so no copying is needed at all. Profile before reaching for `deepcopy` as a default.

## Anti-Pattern

In [ ]:
import copy

# anti-pattern: using deepcopy as a default everywhere out of caution
def process(jobs):
    safe = copy.deepcopy(jobs)   # unnecessary if jobs is flat
    return [j.upper() for j in safe]

# correct: jobs is a flat list of strings — immutable elements
# a slice or .copy() is sufficient and much faster
def process(jobs):
    return [j.upper() for j in jobs]   # strings are immutable, no copy needed

batch = ["job_1", "job_2", "job_3"]
print(process(batch))
print(batch)   # unchanged — strings cannot be mutated

## Interview Signals

- What is the difference between shallow copy and deep copy at the memory level?
- When is a shallow copy safe to use and when does it cause bugs?
- How does `copy.deepcopy` handle circular references without looping forever?
- Name three ways to shallow copy a list in Python.

## Exercise

In [ ]:
import copy

def snapshot_cluster(cluster):
    """
    Takes a cluster state dict and returns an independent snapshot.
    The snapshot must be fully isolated — mutations to either the
    original or the snapshot must not affect the other at any depth.

    Bug: the current implementation uses shallow copy.
    Fix it so all assertions pass.
    """
    return cluster.copy()


cluster = {
    "us-east": {"api": "running", "worker": "running"},
    "eu-west": {"api": "running", "worker": "stopped"}
}

snap = snapshot_cluster(cluster)

snap["us-east"]["api"] = "crashed"
cluster["eu-west"]["worker"] = "restarting"

assert cluster["us-east"]["api"] == "running",      "original must not see snapshot changes"
assert snap["eu-west"]["worker"] == "stopped",      "snapshot must not see original changes"
assert snap is not cluster,                         "must be a different object"
assert snap["us-east"] is not cluster["us-east"],  "inner dicts must be independent"
print("all assertions passed")